# Modelos

Y ahora, esta colaboración revela el corazón (¿o el cerebro?) de la librería de transformadores: los modelos:

https://colab.research.google.com/drive/16sYvEc1lYJDlR6TvjIGMs6bPqraOt7Td?usp=share_link

Esto debería funcionar bien en una caja T4 de bajo costo o gratuita.

# Modelos

Observemos la API de nivel inferior de Transformers: los modelos que envuelven el código de PyTorch para los transformadores mismos.

Este cuaderno puede ejecutarse en un entorno de ejecución T4 gratuito o de bajo costo.

In [ ]:
!pip install -q requests torch bitsandbytes transformers sentencepiece accelerate

In [ ]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch

# Inicia sesión en Hugging Face

1. Si aún no lo has hecho, crea una cuenta gratuita de HuggingFace en https://huggingface.co y ve a Configuración, luego Crea un nuevo token de API, otorgándote permisos de escritura

2. Presiona el ícono de la "llave" en el panel lateral de la izquierda y agrega un nuevo secreto:
`HF_TOKEN = your_token`

3. Ejecuta la celda a continuación para iniciar sesión.

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# Modelos que vamos a utilizar

LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"
PHI3 = "microsoft/Phi-3-mini-4k-instruct"
GEMMA2 = "google/gemma-2-2b-it"
QWEN2 = "Qwen/Qwen2-7B-Instruct" # ejercicio para ti
MIXTRAL = "mistralai/Mixtral-8x7B-Instruct-v0.1" # Si esto no se ajusta a la memoria de tu GPU, prueba con otros del hub

In [ ]:
messages = [
    {"role": "system", "content": "Eres un asistente útil"},
    {"role": "user", "content": "Cuentame un chiste divertido para una sala llena de científicos de datos."}
  ]

# Acceder a Llama 3.1 desde Meta

Para poder usar la fantástica Llama 3.1, Meta requiere que firmes sus términos de servicio.

Visita la página de instrucciones del modelo en Hugging Face:
https://huggingface.co/meta-llama/Meta-Llama-3.1-8B

En la parte superior de la página hay instrucciones sobre cómo aceptar sus términos. Si es posible, debes usar el mismo correo electrónico que tu cuenta de huggingface.

En mi experiencia, la aprobación llega en un par de minutos. Una vez que hayas sido aprobado para cualquier modelo 3.1, se aplica a toda la familia de modelos.

In [ ]:
# Quantization Config - Esto nos permite cargar el modelo en la memoria y utilizar menos memoria.
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
# Tokenizer

tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
# `apply_chat_template` ya no devuelve un tensor plano sino un BatchEncoding: hay que pedirle
# `return_dict=True` y desempaquetarlo con ** al llamar a generate(), o generate() revienta
# con un AttributeError sin mensaje.
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", return_dict=True).to("cuda")

In [ ]:
# The model

model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto",
                                             quantization_config=quant_config)

In [ ]:
memory = model.get_memory_footprint() / 1e6
print(f"Huella de memoria: {memory:,.1f} MB")

In [ ]:
model

In [ ]:
outputs = model.generate(**inputs, max_new_tokens=80)
print(tokenizer.decode(outputs[0]))

In [ ]:
# Limpiar

del inputs, outputs, model
torch.cuda.empty_cache()

In [ ]:
# Envolviendo todo en una función y agregando Streaming

def generate(model, messages):
  tokenizer = AutoTokenizer.from_pretrained(model)
  tokenizer.pad_token = tokenizer.eos_token
  inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", return_dict=True).to("cuda")
  streamer = TextStreamer(tokenizer)
  model = AutoModelForCausalLM.from_pretrained(model, device_map="auto", quantization_config=quant_config)
  outputs = model.generate(**inputs, max_new_tokens=80, streamer=streamer)
  del tokenizer, streamer, model, inputs, outputs
  torch.cuda.empty_cache()

In [ ]:
generate(PHI3, messages)

In [ ]:
messages = [
    {"role": "user", "content": "Cuenta un chiste divertido para una sala llena de científicos de datos."}
  ]
generate(GEMMA2, messages)